In [7]:
# Notebook 03: Final Outputs
#
# Purpose:
# - Load final county scores
# - Produce ranking tables
# - Export app-ready files

In [8]:
import pandas as pd
from pathlib import Path

In [9]:
BASE_DIR = Path.cwd().parent
final_file = BASE_DIR / "data" / "final" / "gc_cpri_scored_counties.csv"

df = pd.read_csv(final_file)
df.head()

,county,registered_voters,total_ballots_cast,turnout_rate,absentee_rate,population_2023,poverty_rate,pct_less_hs,pct_bachelors,unemployment_rate,median_income,non_white_share,rural_urban_code,broadband_access_rate,gc_cpri_score_raw,gc_cpri_score
0,BALDWIN,207643,122542,0.5902,0.0634,253507.0,10.0,8.268600,32.797637,2.3,71704.0,0.082985,3.0,0.91,-4.835058,0.000000
1,CLARKE,19143,11993,0.6265,0.0604,22337.0,19.0,14.731304,15.426531,5.0,44906.0,0.413350,9.0,0.74,2.287146,95.483093
2,CONECUH,9820,6105,0.6217,0.0785,11174.0,26.5,12.513843,13.129076,3.5,36106.0,0.431829,9.0,0.73,2.624067,100.000000
3,ESCAMBIA,28268,15009,0.5310,0.0477,36558.0,21.3,17.467352,12.490686,3.1,47792.0,0.271598,6.0,0.79,0.602831,72.902508
4,MOBILE,322535,176019,0.5457,0.0489,411640.0,16.3,10.991693,25.150224,3.1,54315.0,0.355915,2.0,0.86,-1.659634,42.570997


In [10]:
ranking = df[["county", "gc_cpri_score"]].sort_values("gc_cpri_score", ascending=False)
ranking.head(20)

,county,gc_cpri_score
2,CONECUH,100.000000
1,CLARKE,95.483093
5,MONROE,86.628169
3,ESCAMBIA,72.902508
7,TOTAL (ALL OF ALABAMA),65.896826
6,WASHINGTON,55.084077
4,MOBILE,42.570997
0,BALDWIN,0.000000


In [11]:
df["gc_cpri_score"].describe()

count      8.000000
mean      64.820709
std       32.757505
min        0.000000
25%       51.955807
50%       69.399667
75%       88.841900
max      100.000000
Name: gc_cpri_score, dtype: float64

In [12]:
APP_DATA_DIR = BASE_DIR / "app" / "data"
APP_DATA_DIR.mkdir(parents=True, exist_ok=True)

app_df = df[["county", "gc_cpri_score"]].copy()
app_df.to_csv(APP_DATA_DIR / "gc_cpri_scores_for_app.csv", index=False)
app_df.to_json(APP_DATA_DIR / "gc_cpri_scores_for_app.json", orient="records")

In [13]:
# Cell: Export v2 JSON for the web app
# Includes per-indicator metadata (mean/std for z-scoring, transform info,
# direction, display_format, PC1 loadings), raw-score normalization params,
# and per-county raw indicator values. The web app uses this to recompute
# scores faithfully when the user adjusts indicator weights.

import json
import numpy as np

features = [
    "poverty_rate",
    "unemployment_rate",
    "median_income",
    "pct_less_hs",
    "non_white_share",
    "rural_urban_code",
]

raw_input = pd.read_csv(BASE_DIR / "data" / "merged" / "gc_cpri_model_input.csv")
loadings = pd.read_csv(BASE_DIR / "data" / "final" / "gc_cpri_pca_loadings.csv", index_col=0)

# Reproduce notebook 02 direction-alignment so means/stds line up with
# what StandardScaler actually saw.
aligned = raw_input[features].copy()
for col in ["poverty_rate", "unemployment_rate", "pct_less_hs"]:
    if aligned[col].max() > 1:
        aligned[col] = aligned[col] / 100
aligned["median_income"] = -1 * aligned["median_income"]

# Median-impute (matches notebook 02 SimpleImputer)
medians = aligned.median()
aligned_imputed = aligned.fillna(medians)

# StandardScaler uses ddof=0 (population std)
means = aligned_imputed.mean()
stds = aligned_imputed.std(ddof=0)

# display_format values:
#   "percent_div100"  -> raw value is e.g. 26.5 (read as 26.5%)
#   "percent_share"   -> raw value is e.g. 0.43 (read as 43.0%)
#   "currency_usd"    -> raw value is dollars
#   "integer"         -> show as whole number
#   "decimal"         -> show with two decimals
indicator_meta = {
    "poverty_rate":      {"label": "Poverty Rate",            "direction": "higher_is_riskier", "transform": "div100", "display_format": "percent_div100"},
    "unemployment_rate": {"label": "Unemployment Rate",        "direction": "higher_is_riskier", "transform": "div100", "display_format": "percent_div100"},
    "median_income":     {"label": "Median Household Income",  "direction": "lower_is_riskier",  "transform": "negate", "display_format": "currency_usd"},
    "pct_less_hs":       {"label": "Adults Without HS Diploma","direction": "higher_is_riskier", "transform": "div100", "display_format": "percent_div100"},
    "non_white_share":   {"label": "Non-White Voter Share",    "direction": "higher_is_riskier", "transform": "none",   "display_format": "percent_share"},
    "rural_urban_code":  {"label": "Rural-Urban Code",         "direction": "higher_is_riskier", "transform": "none",   "display_format": "integer"},
}

indicators_payload = []
for f in features:
    indicators_payload.append({
        "id": f,
        "label": indicator_meta[f]["label"],
        "direction": indicator_meta[f]["direction"],
        "transform": indicator_meta[f]["transform"],
        "display_format": indicator_meta[f]["display_format"],
        "mean": float(means[f]),
        "std":  float(stds[f]),
        "median_for_imputation": float(medians[f]),
    })

pc1_loadings = {f: float(loadings.loc[f, "PC1"]) for f in features}

raw_min = float(df["gc_cpri_score_raw"].min())
raw_max = float(df["gc_cpri_score_raw"].max())

counties_payload = []
for _, row in df.iterrows():
    name = str(row["county"])
    if name.startswith("TOTAL"):
        continue
    county_inds = {}
    for f in features:
        match = raw_input.loc[raw_input["county"] == name, f]
        v = match.iloc[0] if len(match) else None
        county_inds[f] = None if (v is None or pd.isna(v)) else float(v)
    counties_payload.append({
        "name": name.title(),
        "indicators_raw": county_inds,
        "gc_cpri_score": float(row["gc_cpri_score"]),
    })

payload = {
    "version": 2,
    "score_method": "PC1 only, min-max normalized to 0-100",
    "indicators": indicators_payload,
    "pc1_loadings": pc1_loadings,
    "score_normalization": {"raw_min": raw_min, "raw_max": raw_max},
    "counties": counties_payload,
}

with open(APP_DATA_DIR / "gc_cpri_scores_for_app.json", "w") as fp:
    json.dump(payload, fp, indent=2)

print(f"Wrote v2 JSON with {len(counties_payload)} counties and {len(features)} indicators")


Wrote v2 JSON with 7 counties and 6 indicators
